In [14]:
# Importing the necessary Libraries
import os
import pandas as pd
import numpy as np
from pathlib import Path

In [15]:
# Loading the Competitive Matches and National Teams data

base_dir = Path.cwd()
while base_dir.name != "World Cup Predictor":
    base_dir = base_dir.parent

competitive_matches_path = base_dir/"data"/"processed"/"clean_competitive_matches.csv"
competitive_matches_df = pd.read_csv(competitive_matches_path)
national_teams_path = base_dir/"data"/"raw"/"national_teams.csv"
national_teams_df = pd.read_csv(national_teams_path)
elo_ratings_path = base_dir/"data"/"raw"/"elo_ratings.csv"
elo_ratings_df = pd.read_csv(elo_ratings_path)

In [3]:
# Converting date column to datetime
competitive_matches_df["date"] = pd.to_datetime(competitive_matches_df["date"])

#  Calculate a decay weight for each match: matches closer to today get weight ~1.0, older matches get exponentially lower weight. Formula: e^(-λ * years_ago) where λ is the decay rate (use 0.1 as a starting value)
today_date = pd.Timestamp.now()
time_delta = (today_date - competitive_matches_df["date"]) / pd.Timedelta(days=365.25) # (Converting to years)
decay_weight = np.exp(-0.1*time_delta)

# Addind decay_weight as a column to competitive_matches_df before splitting into home/away
competitive_matches_df['weight'] = decay_weight

# Segregating into two dataframes - home and away
home_df = competitive_matches_df[['home_team', 'home_score', 'away_score', 'weight']]
away_df = competitive_matches_df[['away_team', 'home_score', 'away_score', 'weight']]

In [4]:
# Renaming the column names

home_df = home_df.rename(columns={
    'home_team':'team',
    'home_score':'goals_for',
    'away_score':'goals_against'
})
away_df = away_df.rename(columns={
    'away_team':'team',
    'away_score':'goals_for',
    'home_score':'goals_against'
})

In [5]:
# Concatenating both dataframes into one

combined_df = pd.concat([home_df, away_df], ignore_index=True)

In [6]:
# Calculating Team Strength by taking the average of goals scored and conceded by all teams

def weighted_mean(group):
    return pd.Series({
        'goals_for': (group['goals_for'] * group['weight']).sum() / group['weight'].sum(),
        'goals_against': (group['goals_against'] * group['weight']).sum() / group['weight'].sum()
    })

team_strength_df = pd.concat([home_df, away_df], ignore_index = True). groupby('team').apply(weighted_mean) 

In [7]:
# Check the team strengths

print(team_strength_df.head())

             goals_for  goals_against
team                                 
Abkhazia      1.806425       0.748552
Afghanistan   1.025497       1.762315
Albania       1.024373       1.160204
Alderney      0.679606       3.930985
Algeria       1.975623       0.871936


In [17]:
# Using National Teams data to factor in the elo ratings, total market value and fifa rankings.
national_teams = national_teams_df[['name', 'total_market_value', 'fifa_ranking']]

# Using log scaling and min-max normalizing to reward the stronger squads.
national_teams['log_market_value'] = np.log(national_teams['total_market_value'])

squad_value = national_teams['log_market_value']
log_min = squad_value.min()
log_max = squad_value.max()
national_teams['squad_value_weight'] = 0.2 + 1.8 * (squad_value - log_min) / (log_max - log_min) # Apply min-max normalization on the log_market_value column scaled to 0.2-1.0

rank = national_teams['fifa_ranking']
rank_max = rank.max()
national_teams['fifa_ranking_weight'] = 0.2 + 1.8 * (1 - (rank-1)) / (rank_max - 1)

elo = elo_ratings_df['elo_rating']
elo_min = elo.min()
elo_max = elo.max()
elo_ratings_df['elo_rating_weight'] = 0.2 + 1.8 * (elo - elo_min) / (elo_max - elo_min)

In [19]:
# Merging the Elo Ratings DataFrame into National teams DataFrame
national_teams = national_teams.reset_index()
national_teams = pd.merge(national_teams, elo_ratings_df[['name','elo_rating_weight']], left_on='name', right_on='name', how='left')
national_teams['elo_rating_weight'] = national_teams['elo_rating_weight'].fillna(0.2)

# Calculating the National teams weight based on all the 3 individual weights
national_teams['weight'] = 0.3 * national_teams['squad_value_weight'] + 0.3 * national_teams['fifa_ranking_weight'] + 0.4 * national_teams['elo_rating_weight']

In [29]:
 print(national_teams[national_teams['name'].isin(['France', 'England'])][['name', 'elo_rating_weight']])

       name  elo_rating_weight
28  England           1.851957
32   France           1.914237


In [33]:
# Merging the National teams DataFrame into Team Strength DataFrame
team_strength_df = team_strength_df.reset_index()
team_strength_df = pd.merge(team_strength_df, national_teams[['name','weight']], left_on='team', right_on='name', how='left')

In [34]:
team_strength_df['weight'] = team_strength_df['weight'].fillna(0.2)

In [38]:
print(national_teams[national_teams['name'].isin(['France', 'England'])]['weight'])
print(national_teams[national_teams['name'].isin(['France', 'England'])][['name', 'squad_value_weight', 'fifa_ranking_weight', 'elo_rating_weight']])
national_teams_df[national_teams_df['name'].isin(['France','England'])][['national_team_id','name', 'team_code','country_code', 'total_market_value']]
national_teams_df[national_teams_df['total_market_value'].isna()]['name']

28   NaN
32   NaN
Name: weight, dtype: float64
       name  squad_value_weight  fifa_ranking_weight  elo_rating_weight
28  England                 NaN             0.182857           1.851957
32   France                 NaN             0.208571           1.914237


28    England
30      Spain
32     France
Name: name, dtype: str

In [22]:
# Apply the weight to goals_for and goals_against
team_strength_df['goals_for'] = team_strength_df['goals_for'] * team_strength_df['weight']
# team_strength_df['goals_against'] = team_strength_df['goals_against'] * team_strength_df['weight']
team_strength_df = team_strength_df[['team','goals_for','goals_against']].set_index('team')

In [23]:
# print(team_strength_df.index[:5])
# print(team_strength_df[250:300])
print(team_strength_df.loc[['Brazil', 'France', 'England', 'Germany', 'Ecuador', 'Panama', 'Iran']])

         goals_for  goals_against
team                             
Brazil    3.140665       0.717796
France    0.079252       0.803133
England   0.089950       0.628298
Germany   4.094805       0.975703
Ecuador   1.531827       1.047638
Panama    0.969707       1.143973
Iran      1.260987       0.678205


In [13]:
# Save the Team Strength dataframe to the processed data folder

team_strength_path = base_dir/"data"/"processed"/"team_strength.csv"
team_strength_df.to_csv(team_strength_path, index = True)